# 04 — FLUX Quickstart (Diffusers)

This notebook follows the FLUX docs pattern with the **smaller FLUX variant** for teaching:
- model: `black-forest-labs/FLUX.1-schnell`
- pipeline: `FluxPipeline.from_pretrained(...)`
- key settings for schnell: `guidance_scale=0.0`, low step count, `max_sequence_length<=256`

> FLUX can still be heavy. CPU offload is enabled to reduce VRAM usage.

## Step 0 — Install (run once if needed)
If imports fail, run this cell, then restart runtime and rerun from top.

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors
print('Install step ready ✅')

## Step 1 — Imports and setup

In [ ]:
import os
import torch
from diffusers import FluxPipeline

os.makedirs('generated_outputs', exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' else torch.float32

print('Device:', device)
print('DType:', dtype)

## Step 2 — Load FLUX.1-schnell
This mirrors the FLUX docs quickstart pattern.

In [ ]:
model_id = 'black-forest-labs/FLUX.1-schnell'

pipe = FluxPipeline.from_pretrained(model_id, torch_dtype=dtype)

if device == 'cuda':
    pipe.enable_model_cpu_offload()
    print('Loaded with CPU offload ✅')
else:
    pipe = pipe.to('cpu')
    print('Loaded on CPU (this may be very slow).')

## Step 3 — First FLUX generation
For **schnell** in docs:
- `guidance_scale=0.0`
- small step count (e.g., 4)
- `max_sequence_length` up to 256

In [ ]:
prompt = 'A cat holding a sign that says hello world'

image = pipe(
    prompt=prompt,
    guidance_scale=0.0,
    height=768,
    width=1360,
    num_inference_steps=4,
    max_sequence_length=256,
).images[0]

image

## Step 4 — Save output

In [ ]:
out_path = 'generated_outputs/flux_schnell_hello_world.png'
image.save(out_path)
print('Saved:', out_path)

## Step 5 — Minimal experiments

In [ ]:
# Try a second prompt with same schnell settings
prompt_2 = 'Tiny astronaut hatching from an egg on the moon, cinematic lighting'
img2 = pipe(
    prompt=prompt_2,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    height=768,
    width=1360,
).images[0]
img2

## Notes from FLUX docs
- `FLUX.1-dev` typically needs more steps (around 50) and guidance around 3.5 for quality.
- `FLUX.1-schnell` is the better classroom default for speed.
- If memory is tight, keep CPU offload enabled and reduce resolution.